In [1]:
import json
import pandas as pd
import re
from pathlib import Path

DATA_DIR = Path("/home/vmadmin/intent/src/test")
DATA_FILE = DATA_DIR / "validated_outputs/qwen3-30b-a3b-instruct-2507.csv"
OUTPUT_FILE = Path("/home/vmadmin/intent/src/results/qwen3-30b-a3b-instruct-2507.csv")


def load_dataset():
    if not DATA_FILE.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {DATA_FILE}")
    df = pd.read_csv(DATA_FILE)
    df["data_source"] = DATA_FILE.name
    return df


def _extract(pattern: str, text: str, cast=None):
    match = re.search(pattern, text, re.S)
    if not match:
        return None
    value = match.group(1).strip()
    if value in {"None", "null"}:
        return None
    return cast(value) if cast else value


def parse_usage_fields(payload: str) -> dict:
    patterns = {
        "prompt_tokens": (r"CompletionUsage\(.*?prompt_tokens=([0-9]+)", int),
        "completion_tokens": (r"CompletionUsage\(completion_tokens=([0-9]+)", int),
        "total_tokens": (r"CompletionUsage\(.*?total_tokens=([0-9]+)", int),
        "reasoning_tokens": (r"CompletionTokensDetails\(.*?reasoning_tokens=([0-9]+)", int),
        "cached_tokens": (r"PromptTokensDetails\(.*?cached_tokens=([0-9]+)", int),
        "cache_write_tokens": (r"PromptTokensDetails\(.*?cache_write_tokens=([0-9]+)", int),
        "cost": (r"CompletionUsage\(.*?\bcost=([0-9.eE+-]+)", float),
    }
    return {key: _extract(pattern, payload, caster) for key, (pattern, caster) in patterns.items()}


def parse_policy_status(policy_column_value: str) -> str:
    if not isinstance(policy_column_value, str):
        return None
    try:
        payload = json.loads(policy_column_value)
    except json.JSONDecodeError:
        return None
    return payload.get("status")


def parse_function_call_name(payload: str) -> str:
    match = re.search(r"ChatCompletionMessageFunctionToolCall\(.*?name='([^']+)'", payload, re.S)
    return match.group(1) if match else None


def parse_response(payload: str):
    if not isinstance(payload, str) or not payload.startswith("ChatCompletion("):
        return None
    parsed = {
        "id": _extract(r"ChatCompletion\(id='([^']+)'", payload),
        "name": parse_function_call_name(payload),
        "finish_reason": _extract(r"Choice\(finish_reason='([^']+)'", payload),
        "model": _extract(r"model='([^']+)'", payload),
        "provider": _extract(r"provider='([^']+)'\)\s*$", payload),
        "created": _extract(r"created=([0-9]+)", payload, int),
    }
    parsed.update(parse_usage_fields(payload))
    return parsed


raw_df = load_dataset()
response_columns = ["intent_processing", "type_definition"]
records = []
for column in response_columns:
    for intent, payload, policy in zip(raw_df["intent"], raw_df[column], raw_df["policy"]):
        parsed = parse_response(payload)
        if parsed:
            parsed.update(
                {
                    "intent": intent,
                    "response_kind": column,
                    "status": parse_policy_status(policy),
                }
            )
            records.append(parsed)

responses_df = pd.DataFrame(records)
responses_df["created"] = pd.to_datetime(responses_df["created"], unit="s")

responses_df = responses_df[
    [
        "intent",
        "response_kind",
        "id",
        "name",
        "finish_reason",
        "model",
        "provider",
        "created",
        "prompt_tokens",
        "cached_tokens",
        "cache_write_tokens",
        "completion_tokens",
        "reasoning_tokens",
        "total_tokens",
        "cost",
        "status",
    ]
].sort_values(["intent", "response_kind"]).reset_index(drop=True)
responses_df

,intent,response_kind,id,name,finish_reason,model,provider,created,prompt_tokens,cached_tokens,cache_write_tokens,completion_tokens,reasoning_tokens,total_tokens,cost,status
0,Create a slice for a harbor area supporting lo...,intent_processing,gen-1791425464-IY7SkeYTmV7llTs1rLcL,create_session,tool_calls,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 02:11:04,1941,0,0,91,0,2032,0.000300,Policy creation failed
1,Create a slice for a harbor area supporting lo...,type_definition,gen-1791425465-BS8lGStbK1szGwMPK8QV,None,stop,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 02:11:05,267,0,0,3,0,270,0.000036,Policy creation failed
2,Create a slice for a mobile esports tournament...,intent_processing,gen-1791426959-FmM8daLPg6l0N2VFp4GI,create_session,tool_calls,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 02:35:59,1944,0,0,100,0,2044,0.000305,Policy creation failed
3,Create a slice for a mobile esports tournament...,type_definition,gen-1791426960-6N3NMelhi4hh3YfhMGY8,None,stop,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 02:36:00,270,0,0,3,0,273,0.000037,Policy creation failed
4,Create a slice for a retail analytics platform...,intent_processing,gen-1791425591-x3IGGqNqyVwyoNudGmfC,create_session,tool_calls,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 02:13:11,1941,0,0,100,0,2041,0.000304,Policy creation failed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,Provision a slice for urban telepresence kiosk...,type_definition,gen-1791427469-06USB2uZO9N1WsHGVI99,None,stop,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 02:44:29,267,0,0,3,0,270,0.000036,Policy creation failed
176,Set up a slice for a shopping mall to improve ...,intent_processing,gen-1791424023-jhfau34Ru3UB5MJ0o3nk,create_session,tool_calls,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 01:47:03,1955,0,0,102,0,2057,0.000307,Policy creation failed
177,Set up a slice for a shopping mall to improve ...,type_definition,gen-1791424024-QKMHOR9mncpqP3XrZyYs,None,stop,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 01:47:04,281,0,0,3,0,284,0.000038,Policy creation failed
178,Set up a slice for remote training sessions us...,intent_processing,gen-1791425432-7RXqcr0SdOi65e5wpeKn,create_session,tool_calls,qwen/qwen3-30b-a3b-instruct-2507,Alibaba,2026-10-08 02:10:32,1942,0,0,99,0,2041,0.000304,Policy creation failed


In [2]:
print("raw_df shape:", raw_df.shape)
print("responses_df shape:", responses_df.shape)
print("unique intents:", responses_df["intent"].nunique())
display(responses_df["response_kind"].value_counts())
display(responses_df["name"].value_counts(dropna=False))
display(responses_df["status"].value_counts(dropna=False))

raw_df shape: (100, 6)
responses_df shape: (180, 16)
unique intents: 99


response_kind
intent_processing    99
type_definition      81
Name: count, dtype: int64

name
create_session    99
None              81
Name: count, dtype: int64

status
Policy creation failed    162
None                       18
Name: count, dtype: int64

In [3]:
responses_df.to_csv(OUTPUT_FILE, index=False, header=True)